In [1]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
# 1) Ler dados
# ================================

db_path = r"C:\Users\beasa\Desktop\AASE\ScreenTimevsMentalWellness.db"
table_main = "ScreenTimevsMentalWellness"

conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {table_main}", conn)
conn.close()

print("Dimensão inicial:", df.shape)

target = "mental_wellness_index_0_100"

if target not in df.columns:
    raise Exception("Coluna alvo não encontrada.")

y = df[target]
mask = y.notna()
df = df[mask]
y = y[mask]

Dimensão inicial: (399, 20)


In [3]:
# 2) Features do Cenário D
#    Só variáveis de ecrã e sono

# ================================

features_D = [
    "screen_time_hours",
    "work_screen_hours",
    "leisure_screen_hours",
    "sleep_hours",
    "sleep_quality_1_5"
]

features_D = [c for c in features_D if c in df.columns]
X = df[features_D].fillna(df[features_D].mean())

print("Features usadas no Cenário D:", features_D)
print("Dimensão do X:", X.shape)


Features usadas no Cenário D: ['screen_time_hours', 'work_screen_hours', 'leisure_screen_hours', 'sleep_hours', 'sleep_quality_1_5']
Dimensão do X: (399, 5)


In [4]:
# 3) Modelos – 5 técnicas
# ================================

scaler = StandardScaler()

models = {
    "LinearRegression": Pipeline([
        ("scaler", scaler),
        ("model", LinearRegression())
    ]),

    "RandomForest": Pipeline([
        ("scaler", scaler),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "DecisionTree": Pipeline([
        ("scaler", scaler),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("scaler", scaler),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4),
            n_estimators=200,
            random_state=42
        ))
    ]),

    "GradientBoosting": Pipeline([
        ("scaler", scaler),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ))
    ])
}


In [5]:
# 4) Cross-Validation 10-fold
# ================================

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print("\n===== CENÁRIO D — 5 Técnicas de Regressão (modelo simples: ecrã + sono) =====\n")

results = []

for name, pipe in models.items():
    print(f"A avaliar modelo: {name}...")
    
    cv = cross_validate(
        pipe, X, y, cv=kfold,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }
    )

    mae = -cv["test_MAE"].mean()
    rmse = -cv["test_RMSE"].mean()
    r2 = cv["test_R2"].mean()

    print(f"Modelo: {name}")
    print(f" MAE:  {mae:.3f}")
    print(f" RMSE: {rmse:.3f}")
    print(f" R2:   {r2:.3f}\n")

    results.append({
        "Cenário": "D",
        "Modelo": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

df_results = pd.DataFrame(results)


===== CENÁRIO D — 5 Técnicas de Regressão (modelo simples: ecrã + sono) =====

A avaliar modelo: LinearRegression...
Modelo: LinearRegression
 MAE:  7.762
 RMSE: 9.863
 R2:   0.752

A avaliar modelo: RandomForest...
Modelo: RandomForest
 MAE:  8.154
 RMSE: 10.559
 R2:   0.713

A avaliar modelo: DecisionTree...
Modelo: DecisionTree
 MAE:  10.908
 RMSE: 14.477
 R2:   0.462

A avaliar modelo: AdaBoost...
Modelo: AdaBoost
 MAE:  9.084
 RMSE: 11.163
 R2:   0.686

A avaliar modelo: GradientBoosting...
Modelo: GradientBoosting
 MAE:  7.936
 RMSE: 10.412
 R2:   0.722



In [6]:
df_results["MAE"] = df_results["MAE"].round(3)
df_results["RMSE"] = df_results["RMSE"].round(3)
df_results["R2"] = df_results["R2"].round(3)

df_results = df_results.sort_values(by="RMSE")

df_results = df_results.rename(columns={
    "Cenário": "Cenário",
    "Modelo": "Modelo",
    "MAE": "MAE",
    "RMSE": "RMSE",
    "R2": "R²"
})

print("\nResultados finais Cenário D:\n")
print(df_results.to_string(index=False, justify="center"))


Resultados finais Cenário D:

Cenário      Modelo        MAE   RMSE    R² 
   D    LinearRegression  7.762  9.863 0.752
   D    GradientBoosting  7.936 10.412 0.722
   D        RandomForest  8.154 10.559 0.713
   D            AdaBoost  9.084 11.163 0.686
   D        DecisionTree 10.908 14.477 0.462
